# MuFu sampler comparison

Compare alternating and mixed sampling with saved random forest results. This notebook reads CSVs only; it does not train models.

Tables report mean ± sample standard deviation across seeds. Bold marks the better **MuFu sampler** for each metric, not statistical significance. Use validation results for sampler selection and test results for reporting.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

AI_DIR = Path.cwd()
if AI_DIR.name == "notebooks":
    AI_DIR = AI_DIR.parent
elif AI_DIR.name != "ai":
    AI_DIR = AI_DIR / "ai"

sampler_results = pd.read_csv(AI_DIR / "outputs/sampler/sweep_results.csv")
rf_results = pd.read_csv(AI_DIR / "data/baseline/results.csv")
rf_results = rf_results[rf_results["model"].isin(["rf_bioactivity", "rf_toxicity"])]
results = pd.concat([sampler_results, rf_results], ignore_index=True)
# Same seed aggregation as baseline.results.summarize_seeds, using only pandas.
group_columns = [
    column for column in results.columns
    if column not in ("seed", "value", "git_commit", "date")
]
summary = results.groupby(group_columns, dropna=False, as_index=False).agg(
    value=("value", "mean"),
    std=("value", "std"),
    n_seeds=("value", "count"),
)

# Show which runs are being compared.
results.groupby("model").agg(
    seeds=("seed", lambda seeds: sorted(seeds.unique())),
    snapshot=("data_sha256", "unique"),
)


In [ ]:
METRICS = [
    ("BACE1 RMSE", "bace1", "pIC50_BACE1", "rmse", "rf_bioactivity", -1),
    ("EGFR RMSE", "egfr", "pIC50_EGFR", "rmse", "rf_bioactivity", -1),
    ("Tox21 AUC", "tox21", "macro_tox21", "roc_auc", "rf_toxicity", 1),
    ("ClinTox AUC", "clintox", "macro_clintox", "roc_auc", "rf_toxicity", 1),
    ("Overall tox AUC", "overall", "macro_overall", "roc_auc", "rf_toxicity", 1),
]
SAMPLERS = ["mufu_sampler_alternating", "mufu_sampler_mixed"]


def metric_result(model, split, dataset, endpoint, metric):
    rows = summary[
        (summary["model"] == model)
        & (summary["split"] == split)
        & (summary["dataset"] == dataset)
        & (summary["endpoint"] == endpoint)
        & (summary["metric"] == metric)
    ]
    if rows.empty:
        return np.nan, np.nan, 0
    row = rows.iloc[0]
    return row["value"], row["std"], int(row["n_seeds"])


def comparison_table(split):
    lines = [
        f"| {split.title()} | Alternating | Mixed | RF |",
        "| --- | --- | --- | --- |",
    ]
    for label, dataset, endpoint, metric, rf_model, direction in METRICS:
        values = [
            metric_result(model, split, dataset, endpoint, metric)
            for model in [*SAMPLERS, rf_model]
        ]
        best = max((value[0] * direction for value in values[:2]))
        formatted = []
        for index, (mean, std, count) in enumerate(values):
            text = f"{mean:.3f} ± {std:.3f}" if count else "—"
            if index < 2 and count and np.isclose(mean * direction, best, rtol=0, atol=1e-12):
                text = f"**{text}**"
            formatted.append(text)
        lines.append("| " + " | ".join([label, *formatted]) + " |")
    return "\n".join(lines)


## Validation comparison


In [ ]:
display(Markdown(comparison_table("val")))

## Test comparison


In [ ]:
test_table = comparison_table("test")
display(Markdown(test_table))
